<a href="https://colab.research.google.com/github/mooch443/dataset-fixer/blob/main/notebooks/02_task_aware_tiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Task-aware image tiling

This tutorial demonstrates `Dataset.tile()` with overlapping, edge-aligned crops and two negative-tile policies. The package applies equivalent task-aware geometry rules to detection, segmentation, pose, and POLO data.

> **AI-generation disclosure:** this project and tutorial are largely AI-generated under human direction and review. Independently validate results for your data.

The example uses official images and YOLO labels from the public [SAWIT dataset](https://github.com/dtnguyen0304/sawit), which declares the [MIT License](https://github.com/dtnguyen0304/sawit/blob/main/LICENSE).

## 1. Install into the active notebook kernel

In [ ]:
import importlib, os, subprocess, sys, tempfile
from pathlib import Path

repo = next((candidate for candidate in [Path('/content/dataset-fixer'), Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / 'pyproject.toml').is_file()), None)
if repo is None:
    repo = Path(tempfile.mkdtemp(prefix='dataset-fixer-source-')) / 'dataset-fixer'
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/mooch443/dataset-fixer.git', str(repo)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)])
sys.path.insert(0, str(repo / 'src'))
sys.path.insert(0, str(repo))
importlib.invalidate_caches()
import dataset_fixer
print('dataset-fixer:', dataset_fixer.__version__, dataset_fixer.__file__)
WORK_ROOT = Path('/content') if Path('/content').is_dir() else Path(tempfile.gettempdir()) / 'dataset-fixer-notebooks'
WORK_ROOT.mkdir(parents=True, exist_ok=True)

## 2. Download the pinned public dataset subset

The downloader retains upstream pixels, filenames, and YOLO labels exactly. It writes the pinned commit, upstream MIT license, and SHA-256 values to `SOURCE.json`.

In [ ]:
from dataset_fixer import Dataset
from examples.download_public_examples import download_sawit_examples

example_root = WORK_ROOT / 'dataset-fixer-public-examples'
paths = download_sawit_examples(example_root, images_per_class=8)
dataset = Dataset.open(paths['fixed'], task='detect', deep=True)
print(dataset)
dataset.visualize(split='val', n=6, seed=42, columns=3)

## 3. Tile with negative crops retained

The grid uses edge-aligned final windows without resizing. Boxes are clipped and retained only when at least 10% of their original area remains. `negative_tiles='all'` keeps empty crops, which can be useful for learning background variation.

In [ ]:
import shutil
all_destination = WORK_ROOT / 'sawit-grid-all-negatives'
if all_destination.exists():
    shutil.rmtree(all_destination)

all_plan = dataset.tile(
    mode='grid',
    tile_size=640,
    overlap=0.20,
    min_area_ratio=0.10,
    negative_tiles='all',
    visualize=True,
)
print(all_plan)
assert all_plan.data_yaml is None and not all_destination.exists()
all_tiles = all_plan.export(destination=all_destination)
print(all_tiles)
all_tiles.visualize(split='val', n=8, seed=42, columns=4)

## 4. Tile with negative crops removed

The crop geometry is unchanged; only empty output tiles are excluded. Comparing provenance counts makes the policy effect explicit.

In [ ]:
positive_destination = WORK_ROOT / 'sawit-grid-positive-only'
if positive_destination.exists():
    shutil.rmtree(positive_destination)

positive_plan = dataset.tile(
    mode='grid', tile_size=640, overlap=0.20, min_area_ratio=0.10,
    negative_tiles='none', visualize=True,
)
positive_tiles = positive_plan.export(destination=positive_destination)
print('all tiles:', len(all_tiles.provenance))
print('positive-only tiles:', len(positive_tiles.provenance))
assert len(positive_tiles.provenance) <= len(all_tiles.provenance)

## 5. Inspect exact source mapping

Every output tile records its immediate parent, ultimate original, crop coordinates, scale, tile index, annotation count, and transformation chain.

In [ ]:
import json
record = next(iter(all_tiles.provenance.values()))
print(json.dumps(record, indent=2)[:2000])
print('effective settings:', all_tiles.settings)
print('training ready:', all_tiles.training_ready)

### Geometry rules for other tasks

- **Segmentation:** polygons are intersected geometrically with each crop.
- **Pose:** outside keypoints become `(0, 0, 0)` and insufficient instances are removed.
- **POLO grid:** points remain only when their complete configured radius circle fits.
- **POLO coverage mode:** randomized crops target per-point coverage and resize to the requested output resolution.